## 1. Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from src.data_loader       import load_etf_data
from src.absorption_ratio  import compute_ar, compute_ar_zscore
from src.features          import build_features
from src.jump_model        import run_walk_forward_cv
from src.backtest          import run_backtest, plot_results
from src.utils             import compute_metrics

# ── Constants ──────────────────────────────────────────────────────────────
DATA_PATH   = './Data/etf_ohlcv_20160301_20251231.csv'
OUTPUT_PATH = './output/'

SECTOR_TICKERS = ['XLB','XLE','XLF','XLI','XLK','XLP','XLRE','XLU','XLV','XLY']
BROAD_TICKERS  = ['SPY','QQQ','RSP']
ALL_TICKERS    = SECTOR_TICKERS + BROAD_TICKERS

AR_WINDOW        = 252
AR_N_COMPONENTS  = 2
AR_ZSCORE_WINDOW = 252

LOOKBACK_YEARS    = 5
LOOKBACK_DAYS     = LOOKBACK_YEARS * 252
LAMBDA_CANDIDATES = [5, 10, 15, 20, 25, 30, 35, 40, 50,
                     60, 70, 80, 100, 120, 150]
JM_N_STATES  = 2
JM_MAX_ITER  = 300
JM_TOL       = 1e-4
JM_N_INIT    = 10

os.makedirs(OUTPUT_PATH, exist_ok=True)
print('Setup complete.')

## 2. Data Loading & Sanity Checks

In [ ]:
adj, log_rets = load_etf_data(DATA_PATH)
print(f'Shape: {adj.shape}')
print(f'Date range: {adj.index[0].date()} → {adj.index[-1].date()}')
print(f'Tickers: {list(adj.columns)}')
display(adj.head(3))
display(log_rets.describe().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for ticker in SECTOR_TICKERS:
    cum = (1 + log_rets[ticker].dropna()).cumprod()
    ax.plot(cum.index, cum.values, linewidth=1, label=ticker, alpha=0.8)

ax.set_title('Cumulative Returns — 10 Sector ETFs', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (rebased to 1)')
ax.legend(ncol=5, fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Absorption Ratio

In [ ]:
ar = compute_ar(log_rets, SECTOR_TICKERS, W=AR_WINDOW, n_components=AR_N_COMPONENTS)
print(f'AR valid observations: {ar.notna().sum()}')
print(ar.describe().round(4))

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(ar.index, ar.values, color='steelblue', linewidth=1.2, label='Absorption Ratio')
ax1.set_title(f'Absorption Ratio (W={AR_WINDOW}, n_components={AR_N_COMPONENTS})', fontsize=13)
ax1.set_xlabel('Date')
ax1.set_ylabel('AR', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
spy_price = adj['SPY']
ax2.plot(spy_price.index, spy_price.values, color='grey',
         linewidth=1, alpha=0.5, label='SPY Price')
ax2.set_ylabel('SPY Price (USD)', color='grey')
ax2.tick_params(axis='y', labelcolor='grey')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
ar_zscore = compute_ar_zscore(ar, fast_window=15, slow_window=AR_ZSCORE_WINDOW)
print(f'AR z-score valid observations: {ar_zscore.notna().sum()}')

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ar_zscore.index, ar_zscore.values, color='darkorange',
        linewidth=1.2, label='AR Z-Score')
ax.axhline(1,  color='red',   linestyle='--', linewidth=1, alpha=0.7, label='+1σ')
ax.axhline(-1, color='green', linestyle='--', linewidth=1, alpha=0.7, label='-1σ')
ax.axhline(0,  color='black', linestyle='-',  linewidth=0.8, alpha=0.5)

ax.set_title('AR Z-Score with ±1σ Bands (Kritzman eq.2)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Z-Score')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Feature Matrix

In [ ]:
features = build_features(adj, log_rets, ar, ar_zscore)
print(f'Feature matrix shape: {features.shape}')
print(f'Date range: {features.index[0].date()} → {features.index[-1].date()}')
display(features.describe().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

corr = features.corr()
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    square=True, linewidths=0.5,
    ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Walk-Forward Regime Detection

In [ ]:
regime = run_walk_forward_cv(
    feature_df=features,
    spy_returns=log_rets['SPY'],
    lookback_days=LOOKBACK_DAYS,
    lambda_candidates=LAMBDA_CANDIDATES,
    n_states=JM_N_STATES,
    max_iter=JM_MAX_ITER,
    tol=JM_TOL,
    n_init=JM_N_INIT,
)
print(f'Regime series length : {len(regime)}')
print(f'Bull % (regime=1)    : {regime.mean():.1%}')
print('Value counts:')
print(regime.value_counts())

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.step(regime.index, regime.values, where='post',
         color='darkgreen', linewidth=1.2, label='Regime (1=Bull)')
ax1.fill_between(regime.index, 0, regime.values,
                 step='post', alpha=0.15, color='darkgreen')
ax1.set_ylabel('Regime', color='darkgreen')
ax1.set_ylim(-0.15, 1.4)
ax1.tick_params(axis='y', labelcolor='darkgreen')

ax2 = ax1.twinx()
spy_oos = adj['SPY'].loc[regime.index[0]:]
ax2.plot(spy_oos.index, spy_oos.values, color='grey',
         linewidth=1, alpha=0.6, label='SPY Price')
ax2.set_ylabel('SPY Price (USD)', color='grey')
ax2.tick_params(axis='y', labelcolor='grey')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('Walk-Forward Regime Signal with SPY Price', fontsize=13)
ax1.set_xlabel('Date')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Backtest Results

In [ ]:
strat_ret, bnh_ret, strat_m, bnh_m = run_backtest(regime, log_rets['SPY'])

metrics_df = pd.DataFrame([strat_m, bnh_m]).set_index('Label')
print('=== Performance Metrics ===')
display(metrics_df.style.format({
    'CAGR':         '{:.2%}',
    'Volatility':   '{:.2%}',
    'Sharpe':       '{:.3f}',
    'Max DD':       '{:.2%}',
    'Calmar':       '{:.3f}',
    'Pct Invested': '{:.1%}',
}))

metrics_df.to_csv(OUTPUT_PATH + 'metrics.csv')
regime.to_csv(OUTPUT_PATH + 'regime_series.csv')

In [ ]:
plot_results(strat_ret, bnh_ret, regime,
             save_path=OUTPUT_PATH + 'equity_curves.png')

## 7. Innovation: AR-Gated Position Sizing

The base strategy is binary: fully invested in a bull regime, in cash otherwise.
The **AR gate** adds a fragility filter on top of the regime signal. On days when
the Jump Model says "bull" but the (lagged) Absorption Ratio z-score is elevated
(systemic risk is rising), exposure is **halved to 0.5** rather than left at 1.0.

This produces three exposure levels:

| Position | Meaning |
|----------|---------|
| `0.0`    | Bear regime → cash |
| `0.5`    | Bull regime but fragile (high AR) → half exposure |
| `1.0`    | Bull regime and stable (low AR) → full exposure |

There is **no look-ahead**: the AR z-score is shifted by one day inside
`run_backtest()`, exactly like the regime signal. Below we stress test the gate
across three thresholds (`ar_threshold` ∈ {0.5, 1.0, 1.5}).

In [ ]:
AR_THRESHOLDS = [0.5, 1.0, 1.5]

gated_returns_by_thr = {}
gate_rows = [strat_m]  # include base (ungated) JM strategy for reference

for thr in AR_THRESHOLDS:
    g_ret, _, g_m, _ = run_backtest(
        regime,
        log_rets['SPY'],
        ar_zscore=ar_zscore,
        ar_threshold=thr,
    )
    g_m['Label'] = f'AR Gate (thr={thr})'
    gated_returns_by_thr[thr] = g_ret
    gate_rows.append(g_m)

gate_rows.append(bnh_m)  # buy & hold for reference

gate_metrics_df = pd.DataFrame(gate_rows).set_index('Label')
print('=== AR-Gated Position Sizing: Threshold Sweep ===')
display(gate_metrics_df.style.format({
    'CAGR':         '{:.2%}',
    'Volatility':   '{:.2%}',
    'Sharpe':       '{:.3f}',
    'Max DD':       '{:.2%}',
    'Calmar':       '{:.3f}',
    'Pct Invested': '{:.1%}',
}))

gate_metrics_df.to_csv(OUTPUT_PATH + 'metrics_ar_gate.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Base JM strategy (binary)
cum_base = (1 + strat_ret).cumprod()
ax.plot(cum_base.index, cum_base.values, color='steelblue',
        linewidth=1.6, label='JM Strategy (base)')

# AR-gated strategy at each threshold
colors = ['darkorange', 'firebrick', 'purple']
for (thr, g_ret), c in zip(gated_returns_by_thr.items(), colors):
    cum_g = (1 + g_ret).cumprod()
    ax.plot(cum_g.index, cum_g.values, color=c, linewidth=1.3,
            label=f'AR Gate (thr={thr})')

# Buy & hold
cum_bnh = (1 + bnh_ret).cumprod()
ax.plot(cum_bnh.index, cum_bnh.values, color='grey',
        linewidth=1.4, alpha=0.8, label='Buy & Hold')

ax.set_title('AR-Gated Position Sizing — Equity Curves by Threshold', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (rebased to 1)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'ar_gate_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Stress Tests

In [ ]:
# ── Stress Test 1: LOOKBACK_YEARS in [3, 5] ──────────────────────────────
stress_lookback_results = {}

for lb_years in [3, 5]:
    lb_days = lb_years * 252
    print(f'Running LOOKBACK_YEARS={lb_years} ({lb_days} days)...')
    regime_lb = run_walk_forward_cv(
        feature_df=features,
        spy_returns=log_rets['SPY'],
        lookback_days=lb_days,
        lambda_candidates=LAMBDA_CANDIDATES,
        n_states=JM_N_STATES,
        max_iter=JM_MAX_ITER,
        tol=JM_TOL,
        n_init=JM_N_INIT,
    )
    sr, bh, sm, bm = run_backtest(regime_lb, log_rets['SPY'])
    stress_lookback_results[f'LB={lb_years}yr'] = {
        'CAGR':    round(sm['CAGR'],    4),
        'Sharpe':  round(sm['Sharpe'],  3),
        'Max DD':  round(sm['Max DD'],  4),
        'Calmar':  round(sm['Calmar'],  3),
        'Bull%':   round(regime_lb.mean(), 3),
    }

stress_lb_df = pd.DataFrame(stress_lookback_results).T
print('\n=== Stress Test: Lookback Window ===')
display(stress_lb_df)

In [ ]:
# ── Stress Test 2: AR_WINDOW in [126, 252] ───────────────────────────────
stress_arw_results = {}

for arw in [126, 252]:
    print(f'Running AR_WINDOW={arw}...')
    ar_arw = compute_ar(log_rets, SECTOR_TICKERS, W=arw, n_components=AR_N_COMPONENTS)
    arz_arw = compute_ar_zscore(ar_arw, fast_window=15, slow_window=AR_ZSCORE_WINDOW)
    feat_arw = build_features(adj, log_rets, ar_arw, arz_arw)
    regime_arw = run_walk_forward_cv(
        feature_df=feat_arw,
        spy_returns=log_rets['SPY'],
        lookback_days=LOOKBACK_DAYS,
        lambda_candidates=LAMBDA_CANDIDATES,
        n_states=JM_N_STATES,
        max_iter=JM_MAX_ITER,
        tol=JM_TOL,
        n_init=JM_N_INIT,
    )
    sr, bh, sm, bm = run_backtest(regime_arw, log_rets['SPY'])
    stress_arw_results[f'AR_W={arw}'] = {
        'CAGR':   round(sm['CAGR'],   4),
        'Sharpe': round(sm['Sharpe'], 3),
        'Max DD': round(sm['Max DD'], 4),
        'Calmar': round(sm['Calmar'], 3),
        'Bull%':  round(regime_arw.mean(), 3),
    }

stress_arw_df = pd.DataFrame(stress_arw_results).T
print('\n=== Stress Test: AR Window ===')
display(stress_arw_df)

In [ ]:
# ── Stress Test 3: AR_N_COMPONENTS in [1, 2, 3] ──────────────────────────
stress_nc_results = {}

for nc in [1, 2, 3]:
    print(f'Running AR_N_COMPONENTS={nc}...')
    ar_nc = compute_ar(log_rets, SECTOR_TICKERS, W=AR_WINDOW, n_components=nc)
    arz_nc = compute_ar_zscore(ar_nc, fast_window=15, slow_window=AR_ZSCORE_WINDOW)
    feat_nc = build_features(adj, log_rets, ar_nc, arz_nc)
    regime_nc = run_walk_forward_cv(
        feature_df=feat_nc,
        spy_returns=log_rets['SPY'],
        lookback_days=LOOKBACK_DAYS,
        lambda_candidates=LAMBDA_CANDIDATES,
        n_states=JM_N_STATES,
        max_iter=JM_MAX_ITER,
        tol=JM_TOL,
        n_init=JM_N_INIT,
    )
    sr, bh, sm, bm = run_backtest(regime_nc, log_rets['SPY'])
    stress_nc_results[f'n_comp={nc}'] = {
        'CAGR':   round(sm['CAGR'],   4),
        'Sharpe': round(sm['Sharpe'], 3),
        'Max DD': round(sm['Max DD'], 4),
        'Calmar': round(sm['Calmar'], 3),
        'Bull%':  round(regime_nc.mean(), 3),
    }

stress_nc_df = pd.DataFrame(stress_nc_results).T
print('\n=== Stress Test: AR N Components ===')
display(stress_nc_df)